In [1]:
#**1.0 Upload SMS Dataset**

In [2]:
import pandas as pd
import re
import matplotlib.pyplot as plt

In [3]:
import os
os.listdir(r"C:\Users\Hasya-ZanrooMY\OneDrive\Desktop\Nexpert_Hackathon\Datasets")

['cleaned_combined_sms_email.xls', 'Phishing_Email.csv', 'spam.csv']

In [4]:
#sms
df_sms = pd.read_csv(
    r"C:\Users\Hasya-ZanrooMY\OneDrive\Desktop\Nexpert_Hackathon\Datasets\spam.csv",
    encoding="latin-1"
)

df_sms = df_sms[['v1', 'v2']]
df_sms = df_sms.rename(columns={
    'v1': 'label',
    'v2': 'text'
})

df_sms['label'] = df_sms['label'].replace({
    'ham': 'normal',
    'spam': 'fraud'
})

df_sms['source'] = 'sms'

In [5]:
#email
df_email = pd.read_csv(r"C:\Users\Hasya-ZanrooMY\OneDrive\Desktop\Nexpert_Hackathon\Datasets\Phishing_Email.csv")

df_email = df_email[['Email Text', 'Email Type']]
df_email = df_email.rename(columns={
    'Email Text': 'text',
    'Email Type': 'label'
})

df_email['label'] = df_email['label'].replace({
    'Safe Email': 'normal',
    'Phishing Email': 'fraud'
})

df_email['source'] = 'email'

In [6]:
#combine ataset

df = pd.concat([df_sms, df_email], ignore_index=True)
df = df[['text', 'label', 'source']]

In [7]:
#check
print(df.shape)
print(df.head())
print(df['label'].value_counts())
print(df['source'].value_counts())

(24222, 3)
                                                text   label source
0  Go until jurong point, crazy.. Available only ...  normal    sms
1                      Ok lar... Joking wif u oni...  normal    sms
2  Free entry in 2 a wkly comp to win FA Cup fina...   fraud    sms
3  U dun say so early hor... U c already then say...  normal    sms
4  Nah I don't think he goes to usf, he lives aro...  normal    sms
label
normal    16147
fraud      8075
Name: count, dtype: int64
source
email    18650
sms       5572
Name: count, dtype: int64


In [8]:
#basic cleaning dataset level, remove redununt data 

# buang missing
df = df.dropna(subset=['text', 'label'])

# buang text kosong
df = df[df['text'].astype(str).str.strip() != '']

# buang duplicate exact rows
df = df.drop_duplicates()

# reset index
df = df.reset_index(drop=True)

In [9]:
print(df.shape)
print(df['label'].value_counts())

(22706, 3)
label
normal    15496
fraud      7210
Name: count, dtype: int64


In [10]:
#simpan text asal 

df['original_text'] = df['text']

In [11]:
#feature engineering before clean text 

# URL
df['has_url'] = df['text'].str.contains(r'http|www|bit\.ly|tinyurl', case=False, na=False)

# suspicious domain
df['suspicious_domain'] = df['text'].str.contains(
    r'\.ru|\.xyz|\.tk|\.top|\.click|\.live|\.shop',
    case=False,
    na=False
)

# email address
df['has_email'] = df['text'].str.contains(r'\S+@\S+', na=False)

# exclamation
df['many_exclamations'] = df['text'].str.contains(r'!{2,}', na=False)

# suspicious symbols
df['has_symbols'] = df['text'].str.contains(r'[!@#$%^&*]', na=False)

# count of suspicious symbols
df['symbol_count'] = df['text'].str.count(r'[!@#$%^&*]')

# OTP pattern (4-6 digit)
df['has_otp'] = df['text'].str.contains(r'\b\d{4,6}\b', na=False)

# lots of numbers
df['many_numbers'] = df['text'].str.contains(r'\d{5,}', na=False)

# money keywords / amounts
df['has_money'] = df['text'].str.contains(
    r'\brm\b|\$|usd|eur|gbp|cash|prize|won|reward',
    case=False,
    na=False
)

# urgency words
df['has_urgent_words'] = df['text'].str.contains(
    r'urgent|immediately|now|asap|limited|act now|verify|suspended',
    case=False,
    na=False
)

# fraud keywords
df['has_fraud_keywords'] = df['text'].str.contains(
    r'otp|click|link|verify|account|bank|winner|free|claim|prize|password',
    case=False,
    na=False
)

# many capital letters
df['many_caps'] = df['text'].str.contains(r'[A-Z]{4,}', na=False)

In [12]:
#text cleaning function, in here we're normalize the data
def clean_text(text):
    text = str(text).lower()

    # normalize url dan email
    text = re.sub(r'http\S+|www\S+|bit\.ly\S*|tinyurl\S*', ' URL ', text)
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)

    # normalize number
    text = re.sub(r'\d+', ' NUMBER ', text)

    # buang punctuation/symbol dari text utama
    text = re.sub(r'[^\w\s]', ' ', text)

    # buang extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [13]:
df['clean_text'] = df['text'].apply(clean_text)

In [14]:
df[['original_text', 'clean_text', 'label', 'source']].head(10)

,original_text,clean_text,label,source
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,normal,sms
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni,normal,sms
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in NUMBER a wkly comp to win fa cup...,fraud,sms
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,normal,sms
4,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...,normal,sms
5,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey there darling it s been NUMBER wee...,fraud,sms
6,Even my brother is not like to speak with me. ...,even my brother is not like to speak with me t...,normal,sms
7,As per your request 'Melle Melle (Oru Minnamin...,as per your request melle melle oru minnaminun...,normal,sms
8,WINNER!! As a valued network customer you have...,winner as a valued network customer you have b...,fraud,sms
9,Had your mobile 11 months or more? U R entitle...,had your mobile NUMBER months or more u r enti...,fraud,sms


In [15]:
#add feature length 
df['char_count'] = df['original_text'].astype(str).apply(len)
df['word_count'] = df['clean_text'].astype(str).apply(lambda x: len(x.split()))

In [16]:
#remove duplicate based on clean_text
df = df.drop_duplicates(subset=['clean_text', 'label'])
df = df.reset_index(drop=True)

In [17]:
#final check dataset
print(df.shape)
print(df['label'].value_counts())
print(df['source'].value_counts())
print(df.isnull().sum())

(22212, 19)
label
normal    15175
fraud      7037
Name: count, dtype: int64
source
email    17116
sms       5096
Name: count, dtype: int64
text                  0
label                 0
source                0
original_text         0
has_url               0
suspicious_domain     0
has_email             0
many_exclamations     0
has_symbols           0
symbol_count          0
has_otp               0
many_numbers          0
has_money             0
has_urgent_words      0
has_fraud_keywords    0
many_caps             0
clean_text            0
char_count            0
word_count            0
dtype: int64


In [18]:
df.head()

,text,label,source,original_text,has_url,suspicious_domain,has_email,many_exclamations,has_symbols,symbol_count,has_otp,many_numbers,has_money,has_urgent_words,has_fraud_keywords,many_caps,clean_text,char_count,word_count
0,"Go until jurong point, crazy.. Available only ...",normal,sms,"Go until jurong point, crazy.. Available only ...",False,False,False,False,False,0,False,False,False,False,False,False,go until jurong point crazy available only in ...,111,20
1,Ok lar... Joking wif u oni...,normal,sms,Ok lar... Joking wif u oni...,False,False,False,False,False,0,False,False,False,False,False,False,ok lar joking wif u oni,29,6
2,Free entry in 2 a wkly comp to win FA Cup fina...,fraud,sms,Free entry in 2 a wkly comp to win FA Cup fina...,False,False,False,False,True,1,True,True,False,False,True,False,free entry in NUMBER a wkly comp to win fa cup...,155,36
3,U dun say so early hor... U c already then say...,normal,sms,U dun say so early hor... U c already then say...,False,False,False,False,False,0,False,False,False,False,False,False,u dun say so early hor u c already then say,49,11
4,"Nah I don't think he goes to usf, he lives aro...",normal,sms,"Nah I don't think he goes to usf, he lives aro...",False,False,False,False,False,0,False,False,False,False,False,False,nah i don t think he goes to usf he lives arou...,61,14


In [19]:
df.head()

,text,label,source,original_text,has_url,suspicious_domain,has_email,many_exclamations,has_symbols,symbol_count,has_otp,many_numbers,has_money,has_urgent_words,has_fraud_keywords,many_caps,clean_text,char_count,word_count
0,"Go until jurong point, crazy.. Available only ...",normal,sms,"Go until jurong point, crazy.. Available only ...",False,False,False,False,False,0,False,False,False,False,False,False,go until jurong point crazy available only in ...,111,20
1,Ok lar... Joking wif u oni...,normal,sms,Ok lar... Joking wif u oni...,False,False,False,False,False,0,False,False,False,False,False,False,ok lar joking wif u oni,29,6
2,Free entry in 2 a wkly comp to win FA Cup fina...,fraud,sms,Free entry in 2 a wkly comp to win FA Cup fina...,False,False,False,False,True,1,True,True,False,False,True,False,free entry in NUMBER a wkly comp to win fa cup...,155,36
3,U dun say so early hor... U c already then say...,normal,sms,U dun say so early hor... U c already then say...,False,False,False,False,False,0,False,False,False,False,False,False,u dun say so early hor u c already then say,49,11
4,"Nah I don't think he goes to usf, he lives aro...",normal,sms,"Nah I don't think he goes to usf, he lives aro...",False,False,False,False,False,0,False,False,False,False,False,False,nah i don t think he goes to usf he lives arou...,61,14


In [20]:
df.to_csv("cleaned_combined_sms_email.csv", index=False)

In [21]:
print(df.shape)
print(df['label'].value_counts())
print(df[['original_text', 'clean_text']].head(5))

(22212, 19)
label
normal    15175
fraud      7037
Name: count, dtype: int64
                                       original_text  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                          clean_text  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in NUMBER a wkly comp to win fa cup...  
3        u dun say so early hor u c already then say  
4  nah i don t think he goes to usf he lives arou...  


In [22]:
#Train Model

In [23]:
#Prepare X, Y used clean_text as main input 

X = df['clean_text']
y = df['label']

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
#Build Model (TF-IDF+Logistic Regression)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        max_features=5000,
        ngram_range=(1,2)
    )),
    ('clf', LogisticRegression(max_iter=1000))  # ✅ ada ()
])

In [26]:
#train model
model.fit(X_train, y_train)

,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [27]:
#evaluate model

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.95836146747693

Classification Report:
               precision    recall  f1-score   support

       fraud       0.96      0.90      0.93      1408
      normal       0.96      0.98      0.97      3035

    accuracy                           0.96      4443
   macro avg       0.96      0.94      0.95      4443
weighted avg       0.96      0.96      0.96      4443


Confusion Matrix:
 [[1274  134]
 [  51 2984]]


In [28]:
test_cases = [
    "Congratulations! You have won RM5000. Click here now!",
    "Your account has been suspended. Verify immediately.",
    "Hi bro, malam ni jadi tak?",
    "Please find attached the report for today's meeting",
    "Your OTP is 839201. Do not share with anyone"
]

predictions = model.predict(test_cases)

for text, pred in zip(test_cases, predictions):
    print("Text:", text)
    print("Prediction:", pred)
    print("-" * 50)

Text: Congratulations! You have won RM5000. Click here now!
Prediction: fraud
--------------------------------------------------
Text: Your account has been suspended. Verify immediately.
Prediction: fraud
--------------------------------------------------
Text: Hi bro, malam ni jadi tak?
Prediction: normal
--------------------------------------------------
Text: Please find attached the report for today's meeting
Prediction: normal
--------------------------------------------------
Text: Your OTP is 839201. Do not share with anyone
Prediction: normal
--------------------------------------------------


In [29]:
# Compare model: Naive Bayes

from sklearn.naive_bayes import MultinomialNB

nb_model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', MultinomialNB())
])

nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.950033760972316


In [30]:
print("Logistic Regression:", accuracy_score(y_test, y_pred))
print("Naive Bayes:", accuracy_score(y_test, y_pred_nb))

Logistic Regression: 0.95836146747693
Naive Bayes: 0.950033760972316


In [31]:
import pickle

with open("fraud_model_lr.pkl", "wb") as f:
    pickle.dump(model, f)